In [1]:
import numpy as np
import tensorflow as tf
import tensorflow.keras as keras
import matplotlib.pyplot as plt
import pickle

In [2]:
from tensorflow.compat.v1 import ConfigProto
from tensorflow.compat.v1 import InteractiveSession
import os
config=tf.compat.v1.ConfigProto()
config.gpu_options.per_process_gpu_memory_fraction = 0.9
config.gpu_options.allow_growth=True
sess=tf.compat.v1.Session(config=config) 

In [3]:
from load_network import load_VGG
from train import WarmUpCosine, CustomWeightDecaySGD
from save_load import load_noise_if_exists, save_layer_outputs_and_labels, load_layer_outputs_and_labels
from weights_bias_cla import load_wb_if_exists
from evaluate_cla import evalu_stream_main_selected, per_group_ovr_accuracy, evalu_select

In [4]:
with open('data.pkl', 'rb') as f:
    [x_train,y_train,x_test,y_test]=pickle.load(f)
y_train_onehot=tf.keras.utils.to_categorical(y_train,num_classes=4)
y_test_onehot=tf.keras.utils.to_categorical(y_test,num_classes=4)

In [5]:
model=load_VGG()

In [6]:
pred_model = tf.keras.Model(
    inputs=model.get_layer("re_lu_9").output,
    outputs=model.output
)

In [7]:
l_label = [2,6,10,13,17,20,24,27,32,35]

In [8]:
layer_list = [model.layers[l].name for l in l_label]

In [9]:
NOISE_DIR = "./noise/"
save_layer_outputs_and_labels(model, x_train, y_train, layer_list, save_dir="D:/Data_1/VGG-11/layer_cache/base")
for i in range(10):
    NOISE_I_DIR = NOISE_DIR + str(i)
    x_gauss, x_salt, x_move, x_occ = load_noise_if_exists(NOISE_I_DIR)
    save_layer_outputs_and_labels(model, x_gauss, y_train, layer_list, save_dir="D:/Data_1/VGG-11/layer_cache/gauss/"+str(i))
    save_layer_outputs_and_labels(model, x_salt, y_train, layer_list, save_dir="D:/Data_1/VGG-11/layer_cache/salt/"+str(i))
    save_layer_outputs_and_labels(model, x_move, y_train, layer_list, save_dir="D:/Data_1/VGG-11/layer_cache/move/"+str(i))
    save_layer_outputs_and_labels(model, x_occ, y_train, layer_list, save_dir="D:/Data_1/VGG-11/layer_cache/occ/"+str(i))

[SAVE] Generating layer outputs for: D:/Data_1/VGG-11/layer_cache/base
[Saved] re_lu: outputs (20000, 32768), labels (20000, 1)
[Saved] re_lu_1: outputs (20000, 8192), labels (20000, 1)
[Saved] re_lu_2: outputs (20000, 4096), labels (20000, 1)
[Saved] re_lu_3: outputs (20000, 4096), labels (20000, 1)
[Saved] re_lu_4: outputs (20000, 2048), labels (20000, 1)
[Saved] re_lu_5: outputs (20000, 2048), labels (20000, 1)
[Saved] re_lu_6: outputs (20000, 1024), labels (20000, 1)
[Saved] re_lu_7: outputs (20000, 1024), labels (20000, 1)
[Saved] re_lu_8: outputs (20000, 256), labels (20000, 1)
[Saved] re_lu_9: outputs (20000, 256), labels (20000, 1)
[SAVE] Generating layer outputs for: D:/Data_1/VGG-11/layer_cache/gauss/0
[Saved] re_lu: outputs (20000, 32768), labels (20000, 1)
[Saved] re_lu_1: outputs (20000, 8192), labels (20000, 1)
[Saved] re_lu_2: outputs (20000, 4096), labels (20000, 1)
[Saved] re_lu_3: outputs (20000, 4096), labels (20000, 1)
[Saved] re_lu_4: outputs (20000, 2048), labels 

In [10]:
for i in range(5):
    path = "./attack/" + str(i)
    ATTACK_DIR = os.path.join(path, "VGG_pgd.npy")
    x_attack = np.load(ATTACK_DIR)
    save_layer_outputs_and_labels(model, x_attack, y_train, layer_list, save_dir="D:/Data_1/VGG-11/layer_cache/attack/"+str(i))

[SAVE] Generating layer outputs for: D:/Data_1/VGG-11/layer_cache/attack/0
[Saved] re_lu: outputs (20000, 32768), labels (20000, 1)
[Saved] re_lu_1: outputs (20000, 8192), labels (20000, 1)
[Saved] re_lu_2: outputs (20000, 4096), labels (20000, 1)
[Saved] re_lu_3: outputs (20000, 4096), labels (20000, 1)
[Saved] re_lu_4: outputs (20000, 2048), labels (20000, 1)
[Saved] re_lu_5: outputs (20000, 2048), labels (20000, 1)
[Saved] re_lu_6: outputs (20000, 1024), labels (20000, 1)
[Saved] re_lu_7: outputs (20000, 1024), labels (20000, 1)
[Saved] re_lu_8: outputs (20000, 256), labels (20000, 1)
[Saved] re_lu_9: outputs (20000, 256), labels (20000, 1)
[SAVE] Generating layer outputs for: D:/Data_1/VGG-11/layer_cache/attack/1
[Saved] re_lu: outputs (20000, 32768), labels (20000, 1)
[Saved] re_lu_1: outputs (20000, 8192), labels (20000, 1)
[Saved] re_lu_2: outputs (20000, 4096), labels (20000, 1)
[Saved] re_lu_3: outputs (20000, 4096), labels (20000, 1)
[Saved] re_lu_4: outputs (20000, 2048), la

In [11]:
CACHE_DIR = "./VGG-11/w_and_b_cache"

In [12]:
eva_w,eva_b = load_wb_if_exists(y_train, layer_list, CACHE_DIR, save_dir="D:/Data_1/VGG-11/layer_cache/base")

In [13]:
NOISE_dir='./noise/'
ATTACK_dir=Noise_dir='./attack/'

In [14]:
select_list = evalu_select(layer_list, y_train, eva_w, eva_b, pred_model=pred_model, save_dir = "D:/Data_1/VGG-11/layer_cache/base")

[3 0 2 ... 3 2 2]
[3 0 2 ... 3 2 2]
[3 0 2 ... 3 2 2]
[3 0 2 ... 3 2 2]


In [15]:
base = evalu_stream_main_selected(layer_list, y_train, eva_w, eva_b, select_list, save_dir = "D:/Data_1/VGG-11/layer_cache/base")
base_final = per_group_ovr_accuracy(model, x_train, y_train, select_list)
base = np.concatenate((base,base_final.reshape(1,4)),axis=0)

Layer 0
accuracy: [0.65504417 0.72552882 0.57650526 0.73203694]
Layer 1
accuracy: [0.68596702 0.74977032 0.73005365 0.76406737]
Layer 2
accuracy: [0.73299793 0.78399353 0.77907336 0.78142818]
Layer 3
accuracy: [0.7986843  0.80976635 0.85968648 0.78001685]
Layer 4
accuracy: [0.80972041 0.88850774 0.90590614 0.83417904]
Layer 5
accuracy: [0.86448033 0.90449849 0.94452871 0.86848741]
Layer 6
accuracy: [0.96726039 0.95750691 0.97466073 0.96875773]
Layer 7
accuracy: [0.93826895 0.92850728 0.95252834 0.94150463]
Layer 8
accuracy: [0.99624344 0.99874873 0.99698999 0.99347416]
Layer 9
accuracy: [0.96523191 0.94475056 0.94172571 0.91247641]


In [16]:
base

array([[0.65504417, 0.72552882, 0.57650526, 0.73203694],
       [0.68596702, 0.74977032, 0.73005365, 0.76406737],
       [0.73299793, 0.78399353, 0.77907336, 0.78142818],
       [0.7986843 , 0.80976635, 0.85968648, 0.78001685],
       [0.80972041, 0.88850774, 0.90590614, 0.83417904],
       [0.86448033, 0.90449849, 0.94452871, 0.86848741],
       [0.96726039, 0.95750691, 0.97466073, 0.96875773],
       [0.93826895, 0.92850728, 0.95252834, 0.94150463],
       [0.99624344, 0.99874873, 0.99698999, 0.99347416],
       [0.96523191, 0.94475056, 0.94172571, 0.91247641],
       [1.        , 1.        , 1.        , 1.        ]])

In [17]:
attack = np.zeros((len(layer_list),4))
for i in range(5):
    attack += evalu_stream_main_selected(layer_list, y_train, eva_w, eva_b, select_list, save_dir = "D:/Data_1/VGG-11/layer_cache/attack/"+str(i))
attack = attack/5
attack_final = np.zeros(4)
for i in range(5):
    DIR = ATTACK_dir + str(i) + '/VGG_pgd.npy'
    x_noise = np.load(DIR)
    attack_final += per_group_ovr_accuracy(model, x_noise, y_train, select_list)
attack_final = attack_final/5
attack = np.concatenate((attack,attack_final.reshape(1,4)),axis=0)

Layer 0
accuracy: [0.62153392 0.69828292 0.63971796 0.69542169]
Layer 1
accuracy: [0.74554482 0.74976356 0.74058162 0.76431858]
Layer 2
accuracy: [0.75020881 0.69598243 0.71888927 0.57175871]
Layer 3
accuracy: [0.66394708 0.6137487  0.51646653 0.38910942]
Layer 4
accuracy: [0.65300434 0.49674071 0.43256441 0.32708559]
Layer 5
accuracy: [0.52872676 0.48074654 0.37286327 0.30026079]
Layer 6
accuracy: [0.46910671 0.46774408 0.28071396 0.25601936]
Layer 7
accuracy: [0.38855162 0.53524031 0.35513102 0.3082414 ]
Layer 8
accuracy: [0.50378754 0.39124324 0.39011432 0.2767153 ]
Layer 9
accuracy: [0.51628147 0.48423855 0.41344337 0.28969615]
Layer 0
accuracy: [0.62105044 0.69854045 0.63997882 0.69272508]
Layer 1
accuracy: [0.74381838 0.74501273 0.74073199 0.76186543]
Layer 2
accuracy: [0.74724152 0.69223261 0.72097606 0.57256563]
Layer 3
accuracy: [0.66789367 0.62100234 0.52250148 0.39727245]
Layer 4
accuracy: [0.65047486 0.49622766 0.43555474 0.33920727]
Layer 5
accuracy: [0.52242096 0.49001358

In [18]:
attack

array([[0.62130112, 0.69803582, 0.63944996, 0.69374438],
       [0.74386012, 0.74671636, 0.74103419, 0.76482423],
       [0.74872598, 0.69012109, 0.72087529, 0.57134474],
       [0.66498082, 0.61470319, 0.51827862, 0.39023214],
       [0.65151353, 0.49854213, 0.42994585, 0.33462196],
       [0.52668785, 0.48480286, 0.3702443 , 0.30628604],
       [0.47052479, 0.46683354, 0.2820828 , 0.25777179],
       [0.38289857, 0.53694119, 0.34602224, 0.31081201],
       [0.49899641, 0.39270295, 0.38710527, 0.27554949],
       [0.51354435, 0.48560136, 0.41037464, 0.29039839],
       [0.51620001, 0.46084999, 0.40204999, 0.28975   ]])

In [19]:
gauss = np.zeros((len(layer_list),4))
for i in range(10):
    gauss += evalu_stream_main_selected(layer_list, y_train, eva_w, eva_b, select_list, save_dir = "D:/Data_1/VGG-11/layer_cache/gauss/"+str(i))
gauss = gauss/10
gauss_final = np.zeros(4)
for i in range(10):
    DIR = NOISE_dir + str(i) + '/gauss.npy'
    x_noise = np.load(DIR)
    gauss_final += per_group_ovr_accuracy(model, x_noise, y_train, select_list)
gauss_final = gauss_final/10
gauss = np.concatenate((gauss,gauss_final.reshape(1,4)),axis=0)

Layer 0
accuracy: [0.25093866 0.24997791 0.7500791  0.25000266]
Layer 1
accuracy: [0.7503169  0.74581151 0.74982451 0.71160411]
Layer 2
accuracy: [0.7500669  0.75726841 0.7500791  0.73576097]
Layer 3
accuracy: [0.75931982 0.42803964 0.76289681 0.25474127]
Layer 4
accuracy: [0.7550562  0.75902069 0.75604112 0.50982655]
Layer 5
accuracy: [0.76181143 0.75576415 0.76446814 0.74183145]
Layer 6
accuracy: [0.76702828 0.76376801 0.76871288 0.63163692]
Layer 7
accuracy: [0.77578749 0.75750993 0.76777449 0.74642848]
Layer 8
accuracy: [0.78726226 0.75827295 0.76691112 0.51716614]
Layer 9
accuracy: [0.77201802 0.75501914 0.75827323 0.62119852]
Layer 0
accuracy: [0.25119234 0.24997791 0.7500791  0.25000266]
Layer 1
accuracy: [0.75031968 0.7485713  0.74982451 0.70788176]
Layer 2
accuracy: [0.7500669  0.75600643 0.7500791  0.72383494]
Layer 3
accuracy: [0.75830224 0.42527208 0.76071275 0.25477913]
Layer 4
accuracy: [0.7545589  0.76176851 0.75679156 0.48946155]
Layer 5
accuracy: [0.76106352 0.75701816

In [20]:
gauss

array([[0.25083974, 0.24997791, 0.7500791 , 0.25000266],
       [0.75026885, 0.74505274, 0.74982451, 0.71217159],
       [0.7500669 , 0.7561174 , 0.75005364, 0.7355868 ],
       [0.75734267, 0.42467837, 0.7602404 , 0.25501767],
       [0.75388793, 0.76202198, 0.75566212, 0.49269823],
       [0.76273391, 0.75679661, 0.76572128, 0.74447012],
       [0.76799869, 0.76504612, 0.76919874, 0.63458447],
       [0.77469553, 0.75739199, 0.76570318, 0.74334346],
       [0.78791227, 0.75926798, 0.76580448, 0.50624897],
       [0.77039615, 0.75379432, 0.75780621, 0.61401559],
       [0.7889    , 0.75599999, 0.76275   , 0.41435   ]])

In [21]:
salt = np.zeros((len(layer_list),4))
for i in range(10):
    salt += evalu_stream_main_selected(layer_list, y_train, eva_w, eva_b, select_list, save_dir = "D:/Data_1/VGG-11/layer_cache/salt/"+str(i))
salt = salt/10
salt_final = np.zeros(4)
for i in range(10):
    DIR = NOISE_dir + str(i) + '/salt.npy'
    x_noise = np.load(DIR)
    salt_final += per_group_ovr_accuracy(model, x_noise, y_train, select_list)
salt_final = salt_final/10
salt = np.concatenate((salt,salt_final.reshape(1,4)),axis=0)

Layer 0
accuracy: [0.33560459 0.29448355 0.7488291  0.27000141]
Layer 1
accuracy: [0.5126703  0.24997791 0.58966665 0.25000266]
Layer 2
accuracy: [0.75403909 0.663754   0.74930993 0.5836521 ]
Layer 3
accuracy: [0.52138619 0.34273342 0.7540977  0.28252495]
Layer 4
accuracy: [0.76682062 0.77875024 0.75582116 0.75337889]
Layer 5
accuracy: [0.72669151 0.76801812 0.75655657 0.78280039]
Layer 6
accuracy: [0.61910712 0.79077116 0.76559786 0.77436558]
Layer 7
accuracy: [0.69014538 0.77226542 0.75733324 0.78702892]
Layer 8
accuracy: [0.64462309 0.79378244 0.77204574 0.76871123]
Layer 9
accuracy: [0.70367586 0.76601584 0.75337079 0.77849806]
Layer 0
accuracy: [0.33929022 0.2879832  0.74934787 0.26798965]
Layer 1
accuracy: [0.50898199 0.24997791 0.58502064 0.25000266]
Layer 2
accuracy: [0.75529944 0.66097148 0.74931658 0.57700825]
Layer 3
accuracy: [0.51867806 0.34347211 0.75080379 0.27703329]
Layer 4
accuracy: [0.76348222 0.78147825 0.7530816  0.76468566]
Layer 5
accuracy: [0.73248908 0.77127014

In [22]:
salt

array([[0.33943279, 0.29030731, 0.74921048, 0.26915904],
       [0.51066441, 0.24997791, 0.58621789, 0.25000266],
       [0.75384249, 0.66227519, 0.74979009, 0.58026757],
       [0.5170272 , 0.34210672, 0.75601492, 0.28010025],
       [0.76245317, 0.77933099, 0.75575726, 0.75641102],
       [0.73094853, 0.76865106, 0.75689922, 0.78253885],
       [0.61902566, 0.78491575, 0.76063395, 0.77732101],
       [0.69521729, 0.77219575, 0.75723899, 0.78846521],
       [0.6488043 , 0.79701806, 0.77077648, 0.77365884],
       [0.70896871, 0.76639546, 0.7540923 , 0.78644857],
       [0.61575   , 0.79660001, 0.77479999, 0.7461    ]])

In [23]:
move = np.zeros((len(layer_list),4))
for i in range(10):
    move += evalu_stream_main_selected(layer_list, y_train, eva_w, eva_b, select_list, save_dir = "D:/Data_1/VGG-11/layer_cache/move/"+str(i))
move = move/10
move_final = np.zeros(4)
for i in range(10):
    DIR = NOISE_dir + str(i) + '/move.npy'
    x_noise = np.load(DIR)
    move_final += per_group_ovr_accuracy(model, x_noise, y_train, select_list)
move_final = move_final/10
move = np.concatenate((move,move_final.reshape(1,4)),axis=0)

Layer 0
accuracy: [0.73579393 0.74826982 0.29435276 0.7457605 ]
Layer 1
accuracy: [0.58274771 0.72376223 0.59850573 0.74573373]
Layer 2
accuracy: [0.55496739 0.72850182 0.55013388 0.75595013]
Layer 3
accuracy: [0.76327292 0.75002209 0.76962039 0.74999734]
Layer 4
accuracy: [0.59169998 0.75002209 0.38830709 0.74998512]
Layer 5
accuracy: [0.77901162 0.75002209 0.7255186  0.74999734]
Layer 6
accuracy: [0.65369437 0.75002209 0.42884408 0.75901742]
Layer 7
accuracy: [0.76555094 0.75002209 0.80719597 0.75024684]
Layer 8
accuracy: [0.73644183 0.76102523 0.58082966 0.75350751]
Layer 9
accuracy: [0.76002479 0.75127784 0.66135734 0.73976249]
Layer 0
accuracy: [0.73804927 0.74801731 0.29601055 0.74599954]
Layer 1
accuracy: [0.57822146 0.72375573 0.60175027 0.7469842 ]
Layer 2
accuracy: [0.55475229 0.73600618 0.54442634 0.75422539]
Layer 3
accuracy: [0.7653083  0.75002209 0.7705924  0.74999734]
Layer 4
accuracy: [0.60189702 0.75002209 0.39107118 0.7492407 ]
Layer 5
accuracy: [0.77779426 0.75002209

In [24]:
move

array([[0.73826386, 0.74809239, 0.29429667, 0.74645024],
       [0.57999796, 0.72243483, 0.59875227, 0.7464533 ],
       [0.55786835, 0.73400056, 0.54366687, 0.75513064],
       [0.76393639, 0.75002209, 0.77064086, 0.74999734],
       [0.59835927, 0.75002209, 0.38555718, 0.74987212],
       [0.77900437, 0.75002209, 0.72459209, 0.74999734],
       [0.65700392, 0.75009711, 0.42796272, 0.75831633],
       [0.76762032, 0.75002209, 0.80818814, 0.750171  ],
       [0.74078916, 0.75962   , 0.57699203, 0.75401817],
       [0.76198817, 0.75152106, 0.66506946, 0.74074847],
       [0.6822    , 0.76337499, 0.53185   , 0.753775  ]])

In [25]:
occ = np.zeros((len(layer_list),4))
for i in range(10):
    occ += evalu_stream_main_selected(layer_list, y_train, eva_w, eva_b, select_list, save_dir = "D:/Data_1/VGG-11/layer_cache/occ/"+str(i))
occ = occ/10
occ_final = np.zeros(4)
for i in range(10):
    DIR = NOISE_dir + str(i) + '/occ.npy'
    x_noise = np.load(DIR)
    occ_final += per_group_ovr_accuracy(model, x_noise, y_train, select_list)
occ_final = occ_final/10
occ = np.concatenate((occ,occ_final.reshape(1,4)),axis=0)

Layer 0
accuracy: [0.58202603 0.6925081  0.70192959 0.69626526]
Layer 1
accuracy: [0.63596394 0.59271075 0.68474072 0.60840708]
Layer 2
accuracy: [0.69820056 0.69448296 0.76883546 0.66153224]
Layer 3
accuracy: [0.67567443 0.75651894 0.72796433 0.68607745]
Layer 4
accuracy: [0.77592864 0.79402969 0.85748986 0.77903645]
Layer 5
accuracy: [0.82047285 0.77902349 0.88244986 0.80325256]
Layer 6
accuracy: [0.83242708 0.79300891 0.86046031 0.74877172]
Layer 7
accuracy: [0.84445546 0.7775102  0.88045865 0.80054146]
Layer 8
accuracy: [0.87518656 0.83200293 0.90269155 0.70050581]
Layer 9
accuracy: [0.84716325 0.7792592  0.87344814 0.70153897]
Layer 0
accuracy: [0.57755543 0.68727152 0.70375732 0.69136954]
Layer 1
accuracy: [0.62670916 0.59074449 0.68776048 0.60880365]
Layer 2
accuracy: [0.69266624 0.70423173 0.77549558 0.65703583]
Layer 3
accuracy: [0.66497011 0.76003297 0.71850907 0.66483868]
Layer 4
accuracy: [0.77874528 0.79426218 0.85120437 0.77031515]
Layer 5
accuracy: [0.82229276 0.78051626

In [26]:
occ

array([[0.58016165, 0.68965049, 0.70346221, 0.69689672],
       [0.63357705, 0.59668798, 0.68388334, 0.61058927],
       [0.69958055, 0.705703  , 0.76441794, 0.65996291],
       [0.66775334, 0.75714724, 0.72198749, 0.67897472],
       [0.77639914, 0.79328286, 0.84932782, 0.77748834],
       [0.81967602, 0.77970429, 0.87190783, 0.80151385],
       [0.81841114, 0.79214447, 0.85751331, 0.75235193],
       [0.83864729, 0.77694896, 0.8744866 , 0.80001977],
       [0.86200561, 0.82113235, 0.89377196, 0.69940756],
       [0.841136  , 0.78199734, 0.86640826, 0.70004571],
       [0.860025  , 0.828075  , 0.885725  , 0.68279999]])

In [27]:
SAVE_FILE='VGG-11.pkl'

In [28]:
progress = {"base": base, "attack": attack,"gauss": gauss,"salt": salt,"move": move,"occ": occ}
with open(SAVE_FILE, "wb") as f:
    pickle.dump(progress, f)

In [29]:
def stats_minmax_from_matrix_dict(matrix_dict, check_num=6):
    """
    Input:
        matrix_dict: {name: np.ndarray(m, n)}, 6 matrices total
    Output:
        stats: {
            name: {
                "mean": (m,),
                "min":  (m,),
                "max":  (m,)
            }, ...
        }
    For each component i, compute statistics along axis=1 (across n samples):
        mean[i] = mean(X[i, :])
        min[i]  = min(X[i, :])
        max[i]  = max(X[i, :])
    """
    if check_num is not None and len(matrix_dict) != check_num:
        raise ValueError(f"Expected {check_num} matrices, got {len(matrix_dict)}")

    # shape consistency
    shapes = [np.asarray(v).shape for v in matrix_dict.values()]
    if len(set(shapes)) != 1:
        raise ValueError(f"Matrix shapes inconsistent: {shapes}")

    stats = {}
    for name, X in matrix_dict.items():
        X = np.asarray(X)
        stats[name] = {
            "mean": X.mean(axis=1),
            "std":  X.std(axis=1),
            "min":  X.min(axis=1),
            "max":  X.max(axis=1),
        }
    return stats

In [31]:
mean_var = stats_minmax_from_matrix_dict(progress)

In [32]:
mean_var

{'base': {'mean': array([0.6722788 , 0.73246459, 0.76937325, 0.8120385 , 0.85957833,
         0.89549873, 0.96704644, 0.9402023 , 0.99636408, 0.94104615,
         1.        ]),
  'std': array([0.0630004 , 0.02943663, 0.02107327, 0.02949243, 0.03909639,
         0.03231365, 0.00616349, 0.00857523, 0.00190029, 0.01881067,
         0.        ]),
  'min': array([0.57650526, 0.68596702, 0.73299793, 0.78001685, 0.80972041,
         0.86448033, 0.95750691, 0.92850728, 0.99347416, 0.91247641,
         1.        ]),
  'max': array([0.73203694, 0.76406737, 0.78399353, 0.85968648, 0.90590614,
         0.94452871, 0.97466073, 0.95252834, 0.99874873, 0.96523191,
         1.        ])},
 'attack': {'mean': array([0.66313282, 0.74910873, 0.68276677, 0.54704869, 0.47865587,
         0.42200526, 0.36930323, 0.3941685 , 0.38858853, 0.42497969,
         0.4172125 ]),
  'std': array([0.03341428, 0.0092931 , 0.06758667, 0.10476673, 0.11553535,
         0.08799449, 0.09975549, 0.08628067, 0.07903615, 0.0863